# Тур по gold-слою платформы

Готовый пример: как подключиться к данным, что где лежит и как строить графики —
**11 визуализаций** уровня футбольной аналитики на живых данных платформы.
gold — витрины, готовые к анализу (read-only). Запусти ячейки по порядку (Shift+Enter).

**Всё параметризовано:** в ячейке «Параметры тура» задаются сезон, команда, соперник
и игрок. Поменяй их и перезапусти ноутбук (Run → Run All Cells) — все графики
пересчитаются сами.

**Каталог данных (OpenMetadata):** описания всех таблиц и колонок, происхождение (lineage) —
[meta.sk-vpn-2026.uk](https://meta.sk-vpn-2026.uk) (нужен VPN).
Вход в каталог: `admin@openmetadata.org` / `admin` (общий аккаунт, только просмотр).
Прямая ссылка на таблицу удара:
[fct_shot в каталоге](https://meta.sk-vpn-2026.uk/table/trino_iceberg.iceberg.gold.fct_shot).
FQN любой таблицы: `trino_iceberg.iceberg.gold.<имя>`.

## 1. Подключение к Trino
Логин/пароль и адрес уже прописаны в окружении контейнера (общий read-only аккаунт).
Писать в данные нельзя — только SELECT из `silver`/`gold`. Здесь же — единый тёмный
стиль для всех графиков ноутбука.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sqlalchemy import create_engine
from trino.auth import BasicAuthentication

# Тёмная палитра в духе футбольной аналитики — одна на весь ноутбук
BG, FG, MUT, GRID = "#0d1520", "#e8ecf1", "#8b95a7", "#1f2b40"
BLUE, TEAL, MAGENTA, YELLOW = "#4da3ff", "#3ec6b4", "#e8497f", "#f2c94c"
RED, GREEN, GREY = "#ff5a5f", "#31c98e", "#6e7891"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": FG, "axes.labelcolor": FG, "axes.titlecolor": FG,
    "xtick.color": MUT, "ytick.color": MUT,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#3a4761",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.9,
    "legend.frameon": False,
})

# SQLAlchemy-движок (pandas.read_sql хочет именно его — иначе UserWarning)
engine = create_engine(
    f"trino://{os.environ['TRINO_USER']}@{os.environ['TRINO_HOST']}:{os.environ['TRINO_PORT']}/iceberg",
    connect_args={
        "auth": BasicAuthentication(os.environ["TRINO_USER"], os.environ["TRINO_PASSWORD"]),
        "http_scheme": "https",
    },
)

def q(sql):
    """Выполнить SQL и вернуть DataFrame."""
    return pd.read_sql(sql, engine)

q("SELECT 1 AS ok")

## 2. Параметры тура
Один блок управляет всеми графиками ниже. Данные тура — АПЛ, сезоны `'1617'`…`'2526'`
(`'2425'` = 2024/25). Имена — как в `dim_team.team_name` / `dim_player.player_name`,
можно частично: `"Liver"` найдёт Liverpool. Поменял — перезапусти ноутбук целиком.

In [ ]:
# ─── ПАРАМЕТРЫ ──────────────────────────────────────────────────────────────
SEASON   = "2425"        # сезон АПЛ: '1617' … '2526'
TEAM     = "Liverpool"   # команда тура
OPPONENT = "Brentford"   # соперник — вместе с TEAM задаёт «матч тура»
PLAYER   = ""            # игрок тура; "" = лучший бомбардир TEAM в сезоне
# ────────────────────────────────────────────────────────────────────────────
SEASON_LBL = f"20{SEASON[:2]}/{SEASON[2:]}"

def team_id(name):
    """team_id по имени команды (можно частичному)."""
    d = q(f"""SELECT team_id, team_name FROM iceberg.gold.dim_team
              WHERE lower(team_name) LIKE lower('%{name}%')""")
    assert len(d) > 0, f"команда «{name}» не найдена в dim_team"
    assert len(d) == 1, f"уточни имя, нашлось несколько: {', '.join(d.team_name)}"
    return d.team_id.iloc[0]

TEAM_ID = team_id(TEAM)

# матч тура: TEAM против OPPONENT в SEASON (если игрались оба круга — последний)
_m = q(f"""
    SELECT m.match_id, m.match_date, m.home_score, m.away_score,
           m.home_team_id, m.away_team_id,
           th.team_name AS home_team, ta.team_name AS away_team
    FROM iceberg.gold.dim_match m
    JOIN iceberg.gold.dim_team th ON m.home_team_id = th.team_id
    JOIN iceberg.gold.dim_team ta ON m.away_team_id = ta.team_id
    WHERE m.season = '{SEASON}' AND m.is_completed
      AND '{TEAM_ID}' IN (m.home_team_id, m.away_team_id)
      AND (lower(th.team_name) LIKE lower('%{OPPONENT}%')
           OR lower(ta.team_name) LIKE lower('%{OPPONENT}%'))
    ORDER BY m.match_date DESC
""")
assert len(_m) > 0, f"матч {TEAM} — {OPPONENT} в сезоне {SEASON} не найден"
MATCH = _m.iloc[0]
TITLE_MATCH = f"{MATCH.home_team} {MATCH.home_score}–{MATCH.away_score} {MATCH.away_team}"

# игрок тура: указанный (ищем по всей лиге) или лучший бомбардир TEAM
_flt = (f"lower(p.player_name) LIKE lower('%{PLAYER}%')" if PLAYER
        else f"s.team_id = '{TEAM_ID}'")
_p = q(f"""
    SELECT p.player_id, p.player_name, p.primary_position, s.team_id
    FROM iceberg.gold.fct_player_season_stats s
    JOIN iceberg.gold.dim_player p ON s.player_id = p.player_id
    WHERE s.season = '{SEASON}' AND {_flt}
    ORDER BY s.goals DESC, s.minutes DESC
""")
assert len(_p) > 0, f"игрок не найден (PLAYER='{PLAYER}')"
PLAYER_ROW = _p.iloc[0]

print(f"Сезон:   {SEASON_LBL}")
print(f"Команда: {TEAM} ({TEAM_ID})")
print(f"Матч:    {TITLE_MATCH} · {MATCH.match_date}")
print(f"Игрок:   {PLAYER_ROW.player_name} ({PLAYER_ROW.primary_position})")

## 3. Что лежит в gold
Две группы таблиц:
- **Словари (`dim_*`)** — справочники: команды, игроки, сезоны, лиги, судьи…
- **Факты (`fct_*`)** — события и метрики: удары, матчи, статистика игроков…

In [ ]:
tables = q("""
    SELECT table_name FROM iceberg.information_schema.tables
    WHERE table_schema = 'gold' ORDER BY 1
""")
dims = tables[tables.table_name.str.startswith('dim_')]
fcts = tables[tables.table_name.str.startswith('fct_')]
print(f"Словари ({len(dims)}):", ", ".join(dims.table_name))
print(f"\nФакты ({len(fcts)}):", ", ".join(fcts.table_name))

## 4. Словари — на примере `dim_team`
Словарь связывает технический `id` с человеческим именем. Пример — команды:

In [ ]:
q("SELECT team_id, team_name, short_name, country FROM iceberg.gold.dim_team ORDER BY team_name").head(10)

Остальные словари устроены так же — `id` + понятные атрибуты:

| Словарь | О чём |
|---|---|
| `dim_player` | игроки (имя, позиция, дата рождения) |
| `dim_competition` | турниры/лиги |
| `dim_season` | сезоны |
| `dim_match` | матчи (дата, команды, счёт) |
| `dim_manager` | тренеры |
| `dim_referee` | судьи |
| `dim_venue` | стадионы |
| `dim_player_attributes` | атрибуты игроков (FIFA-рейтинги) |

Колонки любой таблицы можно посмотреть так:

In [ ]:
q("""
    SELECT column_name, data_type FROM iceberg.information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'dim_player'
    ORDER BY ordinal_position
""")

## 5. Визуализация 1 — карта ударов на макете поля (`fct_shot`)
`fct_shot` (~10 тыс. ударов за сезон) хранит координаты `x`,`y` (0..1 по полю, атака вверх),
ожидаемые голы `xg`, тип атаки `situation` и часть тела `body_part`.

Сначала — два хелпера: **половина поля** (для ударов) и **полное вертикальное поле**
(для карт передач). Пригодятся во всех «полевых» графиках ниже.

In [ ]:
PITCH_L, PITCH_W = 105.0, 68.0  # длина/ширина поля, метры
HALF = 52.5                     # середина поля
PITCH_LINE = "#54627d"          # цвет линий разметки под тёмный фон

def draw_half_pitch(ax, line=PITCH_LINE, lw=1.5):
    """Верхняя половина поля (атака вверх), координаты в метрах."""
    L, W = PITCH_L, PITCH_W
    # внешний контур половины + линия центра
    ax.plot([0, 0, W, W, 0], [HALF, L, L, HALF, HALF], color=line, lw=lw)
    # штрафная площадь (16.5 x 40.32)
    ax.plot([(W-40.32)/2]*2 + [(W+40.32)/2]*2, [L, L-16.5, L-16.5, L], color=line, lw=lw)
    # вратарская (5.5 x 18.32)
    ax.plot([(W-18.32)/2]*2 + [(W+18.32)/2]*2, [L, L-5.5, L-5.5, L], color=line, lw=lw)
    # ворота (7.32)
    ax.plot([(W-7.32)/2, (W+7.32)/2], [L, L], color=line, lw=lw*3, solid_capstyle="butt")
    # точка + дуга пенальти
    ax.scatter([W/2], [L-11], s=9, color=line, zorder=3)
    ax.add_patch(mpatches.Arc((W/2, L-11), 2*9.15, 2*9.15, theta1=217, theta2=323, color=line, lw=lw))
    # центральный круг (верхняя половина) + точка
    ax.add_patch(mpatches.Arc((W/2, HALF), 2*9.15, 2*9.15, theta1=0, theta2=180, color=line, lw=lw))
    ax.scatter([W/2], [HALF], s=7, color=line, zorder=3)
    ax.set_xlim(-2, W+2); ax.set_ylim(HALF-3, L+4)
    ax.set_aspect("equal"); ax.axis("off")

def draw_pitch(ax, line=PITCH_LINE, lw=1.3):
    """Полное вертикальное поле (атака снизу вверх), координаты в метрах."""
    L, W = PITCH_L, PITCH_W
    ax.plot([0, 0, W, W, 0], [0, L, L, 0, 0], color=line, lw=lw)
    ax.plot([0, W], [HALF, HALF], color=line, lw=lw)
    for yy, s in ((L, -1), (0, 1)):          # обе штрафные/вратарские/точки
        ax.plot([(W-40.32)/2]*2 + [(W+40.32)/2]*2,
                [yy, yy+16.5*s, yy+16.5*s, yy], color=line, lw=lw)
        ax.plot([(W-18.32)/2]*2 + [(W+18.32)/2]*2,
                [yy, yy+5.5*s, yy+5.5*s, yy], color=line, lw=lw)
        ax.scatter([W/2], [yy+11*s], s=6, color=line, zorder=3)
    ax.add_patch(mpatches.Circle((W/2, HALF), 9.15, fill=False, color=line, lw=lw))
    ax.add_patch(mpatches.Arc((W/2, L-11), 18.3, 18.3, theta1=217, theta2=323, color=line, lw=lw))
    ax.add_patch(mpatches.Arc((W/2, 11), 18.3, 18.3, theta1=37, theta2=143, color=line, lw=lw))
    ax.set_xlim(-2, W+2); ax.set_ylim(-2, L+2)
    ax.set_aspect("equal"); ax.axis("off")

print("макеты поля готовы")

In [ ]:
# удары TEAM за сезон (для карты) и всей лиги (для тепловой карты)
shots = q(f"""
    SELECT s.y AS wy, s.x AS lx, s.xg, s.is_goal
    FROM iceberg.gold.fct_shot s
    WHERE s.team_id = '{TEAM_ID}' AND s.season = '{SEASON}' AND s.x >= 0.5
""")
league = q(f"""
    SELECT s.y AS wy, s.x AS lx FROM iceberg.gold.fct_shot s
    WHERE s.season = '{SEASON}' AND s.x >= 0.5
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 8))

# --- панель 1: карта ударов, размер = xG, цвет = гол/промах ---
draw_half_pitch(ax1)
goals = shots[shots.is_goal]; miss = shots[~shots.is_goal]
ax1.scatter(miss.wy*PITCH_W, miss.lx*PITCH_L, s=miss.xg*650+15,
            facecolor=GREY, edgecolor=BG, lw=0.5, alpha=0.6, zorder=4)
ax1.scatter(goals.wy*PITCH_W, goals.lx*PITCH_L, s=goals.xg*650+15,
            facecolor=RED, edgecolor="white", lw=0.6, alpha=0.95, zorder=5)
conv = 100*len(goals)/len(shots)
ax1.set_title(f"{TEAM} {SEASON_LBL} — карта ударов")
ax1.text(0, HALF-1.5,
         f"ударов {len(shots)}     голов {len(goals)}     xG {shots.xg.sum():.0f}     реализация {conv:.0f}%",
         ha="left", va="top", fontsize=9.5, color=MUT)
# легенда размеров (xG) в пустой зоне у центра
for xg, xx in [(0.1, 5), (0.3, 14), (0.6, 25)]:
    ax1.scatter([xx], [HALF+3], s=xg*650+15, facecolor="none", edgecolor=MUT, lw=1, zorder=4)
    ax1.text(xx, HALF+6.5, str(xg), ha="center", va="center", fontsize=7.5, color=MUT)
ax1.text(-1, HALF+3, "xG", ha="right", va="center", fontsize=8, color=MUT)
ax1.scatter([], [], facecolor=RED, edgecolor="white", label="гол")
ax1.scatter([], [], facecolor=GREY, edgecolor=BG, label="промах")
ax1.legend(loc="lower right", fontsize=9)

# --- панель 2: тепловая карта — откуда бьёт вся лига ---
H, xe, ye = np.histogram2d(league.wy*PITCH_W, league.lx*PITCH_L,
                           bins=[24, 20], range=[[0, PITCH_W], [HALF, PITCH_L]])
ax2.pcolormesh(xe, ye, H.T, cmap="magma", shading="flat")
draw_half_pitch(ax2, line="#c8cfdb")
ax2.set_title(f"Откуда бьёт вся АПЛ, {SEASON_LBL}  ({len(league):,} ударов)".replace(",", " "))

fig.suptitle("fct_shot — удары на реальном макете поля", fontsize=15, weight="bold", y=0.99)
plt.tight_layout(); plt.show()

## 6. Визуализация 2 — реализация моментов (голы − xG)
xG — сколько голов «должна была» забить команда по качеству моментов. Если забили **больше**
xG — реализуют хладнокровно; **меньше** — транжирят. Считает Trino, join со словарём команд.

In [ ]:
perf = q(f"""
    SELECT t.short_name AS team, sum(s.xg) AS xg, sum(cast(s.is_goal AS int)) AS goals
    FROM iceberg.gold.fct_shot s JOIN iceberg.gold.dim_team t ON s.team_id = t.team_id
    WHERE s.season = '{SEASON}' GROUP BY t.short_name
""")
perf["diff"] = perf.goals - perf.xg
perf = perf.sort_values("diff")
colors = [GREEN if d >= 0 else MAGENTA for d in perf["diff"]]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(perf.team, perf["diff"], color=colors, edgecolor=BG)
ax.axvline(0, color=FG, lw=1)
for i, d in enumerate(perf["diff"]):
    ax.text(d + (0.4 if d >= 0 else -0.4), i, f"{d:+.0f}",
            va="center", ha="left" if d >= 0 else "right", fontsize=8, color=FG)
ax.set_title(f"Реализация моментов: голы минус ожидаемые (xG), {SEASON_LBL}")
ax.set_xlabel("← забивали меньше, чем ожидалось          забивали больше →")
ax.grid(axis="y", visible=False)
ax.margins(x=0.12)
plt.tight_layout(); plt.show()
perf.sort_values("diff", ascending=False).round(1)

## 7. Визуализация 3 — 10 лет эволюции АПЛ (`fct_event`, ~6 млн строк)
Единственный график тура, который смотрит на **всю историю** сразу: один SELECT сканирует
6 млн действий, в ноутбук приезжают 40 чисел. Четыре истории десятилетия: розыгрыш от
ворот стал коротким (правило 2019-го + мода на билд-ап), передачи — точнее, лонгболы
вымирают, прессинг поднялся выше.

In [ ]:
# один скан fct_event: условные агрегаты по типам действий
# гоча Trino: пиши 1e0 (double), а не 1.0 — decimal(2,1) молча съедает точность avg()
evo = q("""
    SELECT season,
           100 * avg(CASE WHEN action = 'pass'
                          THEN if(outcome_success, 1e0, 0e0) END)      AS pass_acc,
           100 * avg(CASE WHEN action = 'pass'
                          THEN if(len_m > 30, 1e0, 0e0) END)           AS long_share,
           100 * avg(CASE WHEN action = 'goalkick'
                          THEN if(len_m < 32, 1e0, 0e0) END)           AS gk_short,
           avg(CASE WHEN action IN ('tackle', 'interception', 'ball_recovery')
                    THEN x END) * 1.05                                  AS def_height_m
    FROM (
        SELECT season, action, outcome_success, x,
               sqrt(pow((end_x - x) * 1.05, 2) + pow((end_y - y) * 0.68, 2)) AS len_m
        FROM iceberg.gold.fct_event
        WHERE action IN ('pass', 'goalkick', 'tackle', 'interception', 'ball_recovery')
    )
    GROUP BY season ORDER BY season
""")
slbl = [f"{s[:2]}/{s[2:]}" for s in evo.season]

PANELS = [
    ("Короткий розыгрыш от ворот, %", "gk_short", TEAL, "{:.0f}",
     "гол-кик короче 32 м: было 1 из 7 — стало каждый второй"),
    ("Точность передач, %", "pass_acc", BLUE, "{:.0f}",
     "лига играет всё чище"),
    ("Длинные передачи (>30 м), %", "long_share", MAGENTA, "{:.0f}",
     "лонгболы вымирают"),
    ("Высота возврата мяча, м от своих ворот", "def_height_m", YELLOW, "{:.1f}",
     "отборы, перехваты и подборы — прессинг поднялся"),
]
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.8))
for ax, (title, col, c, f, sub) in zip(axes.flat, PANELS):
    ys = evo[col].values
    ax.plot(slbl, ys, color=c, lw=2.6, marker="o", ms=5, mec=BG, mew=1)
    ax.fill_between(slbl, ys, ys.min() - (ys.max() - ys.min()) * 0.15, color=c, alpha=0.10)
    for xi, yv in ((0, ys[0]), (len(ys) - 1, ys[-1])):
        ax.annotate(f.format(yv), (xi, yv), textcoords="offset points", xytext=(0, 9),
                    ha="center", color=c, fontsize=10, weight="bold")
    ax.set_title(title, loc="left", pad=17)
    ax.text(0, 1.012, sub, transform=ax.transAxes, fontsize=8.5, color=MUT, va="bottom")
    ax.margins(y=0.22)
    ax.tick_params(axis="x", labelsize=8)
fig.suptitle("Как изменилась АПЛ за 10 сезонов — fct_event, ~6 млн действий",
             fontsize=15, weight="bold")
fig.text(0.5, 0.008, "каждая точка — агрегат ~600 тыс. событий сезона · считает Trino, "
         "в ноутбук приезжает таблица 10×5", ha="center", fontsize=9, color=MUT)
plt.tight_layout(rect=[0, 0.02, 1, 0.99]); plt.show()
evo.round(1)

## 8. Визуализация 4 — xG-флоу матча (ход опасности)
Классика Understat / The Analyst: как по ходу матча копились ожидаемые голы (xG) у обеих
команд. Каждый удар — ступенька вверх, звезда — гол. Сразу видно, кто владел моментами.
Матч задан в параметрах (`TEAM` + `OPPONENT`) — вся его история из `fct_shot`.

In [ ]:
sh = q(f"""SELECT team_id, minute, xg, cast(is_goal AS int) AS goal
           FROM iceberg.gold.fct_shot WHERE match_id = '{MATCH.match_id}' ORDER BY minute""")

FT = max(95, int(sh.minute.max()) + 2)
palette = {MATCH.home_team_id: BLUE, MATCH.away_team_id: RED}
labels  = {MATCH.home_team_id: MATCH.home_team, MATCH.away_team_id: MATCH.away_team}
fig, ax = plt.subplots(figsize=(11, 5))
for tid in (MATCH.home_team_id, MATCH.away_team_id):
    d = sh[sh.team_id == tid].copy()
    d["cum"] = d.xg.cumsum()
    xs = np.concatenate([[0], d.minute.values, [FT]])
    ys = np.concatenate([[0], d.cum.values, [d.cum.iloc[-1] if len(d) else 0]])
    ax.step(xs, ys, where="post", color=palette[tid], lw=2.6,
            label=f"{labels[tid]} — {ys[-1]:.2f} xG")
    g = d[d.goal == 1]
    ax.scatter(g.minute, g.cum, s=190, marker="*", color=palette[tid],
               edgecolor=BG, lw=1.0, zorder=5)

ax.axvline(45, ls="--", color="#3a4761", lw=1)
ax.text(45, ax.get_ylim()[1], " перерыв", ha="left", va="top", fontsize=8.5, color=MUT)
ax.set_title(f"{TITLE_MATCH}    ·    xG-флоу    ·    {MATCH.match_date}")
ax.set_xlabel("минута матча"); ax.set_ylabel("накопленный xG")
ax.set_xlim(0, FT); ax.set_ylim(0, None)
ax.legend(loc="upper left", fontsize=11)
ax.text(0.99, 0.03, "★ — гол", transform=ax.transAxes, ha="right", va="bottom",
        fontsize=9, color=MUT)
plt.tight_layout(); plt.show()

## 9. Визуализация 5 — карта кроссов матча (`fct_event`)
У передач в `fct_event` есть и конечные координаты — можно рисовать стрелки.
Все кроссы **с игры** выбранного матча: цвет команды — дошедшие до партнёра,
маджента — перехваченные, **жёлтые — кроссы под удар** (следующее событие — удар
своей команды). Порядок событий даёт `event_id`, «следующее событие» считает
оконная функция `LEAD()` прямо в Trino.

In [ ]:
cr = q(f"""
    WITH ev AS (
        SELECT team_id, minute, x, y, end_x, end_y, action, outcome_success,
               lead(action)  OVER (PARTITION BY match_id ORDER BY event_id) AS next_action,
               lead(team_id) OVER (PARTITION BY match_id ORDER BY event_id) AS next_team_id
        FROM iceberg.gold.fct_event
        WHERE match_id = '{MATCH.match_id}'
    )
    SELECT * FROM ev WHERE action = 'cross'
""")
cr["is_key"] = (cr.next_action == "shot") & (cr.next_team_id == cr.team_id)

sides = [(MATCH.home_team_id, MATCH.home_team, BLUE),
         (MATCH.away_team_id, MATCH.away_team, TEAL)]
fig, axes = plt.subplots(1, 2, figsize=(12, 10))
for ax, (tid, name, ok_color) in zip(axes, sides):
    draw_pitch(ax)
    d = cr[cr.team_id == tid]
    for _, r in d.iterrows():
        if r.is_key:            color, lw, alpha, z = YELLOW, 2.2, 1.0, 6
        elif r.outcome_success: color, lw, alpha, z = ok_color, 1.8, 0.95, 5
        else:                   color, lw, alpha, z = MAGENTA, 1.1, 0.5, 4
        ax.annotate("", xy=(r.end_y/100*PITCH_W, r.end_x/100*PITCH_L),
                    xytext=(r.y/100*PITCH_W, r.x/100*PITCH_L), zorder=z,
                    arrowprops=dict(arrowstyle="->", color=color, lw=lw, alpha=alpha,
                                    shrinkA=0, shrinkB=0))
        if r.is_key:
            ax.text(r.y/100*PITCH_W, r.x/100*PITCH_L - 1.5, f"{int(r.minute)}'",
                    fontsize=8, color=YELLOW, ha="center", va="top", zorder=6)
    ok, tot = int(d.outcome_success.sum()), len(d)
    pct = 100*ok/tot if tot else 0
    ax.set_title(f"{name}: {ok} / {tot} ({pct:.0f}%)", color=ok_color)

fig.suptitle(f"Кроссы с игры · {TITLE_MATCH}", fontsize=15, weight="bold", y=0.98)
fig.text(0.5, 0.015,
         "угловые и штрафные не входят · жёлтая стрелка — кросс под удар · атака вверх",
         ha="center", fontsize=9, color=MUT)
plt.tight_layout(rect=[0, 0.03, 1, 0.96]); plt.show()

## 10. Визуализация 6 — удары и проникновения к воротам
Послематчевый разбор в одну картинку: карта ударов + **deep entries** — успешные передачи,
доставившие мяч в ~20 м от ворот (пунктирная дуга). Удары — из `fct_shot` (звезда = гол,
размер = xG), проникновения — из `fct_event` (пас — сплошная стрелка, кросс — пунктирная).

In [ ]:
R = 20.0  # радиус «опасной зоны» у ворот, метры
sh2 = q(f"""SELECT team_id, y AS wy, x AS lx, xg, is_goal
            FROM iceberg.gold.fct_shot WHERE match_id = '{MATCH.match_id}'""")
de = q(f"""
    SELECT team_id, x, y, end_x, end_y, action
    FROM iceberg.gold.fct_event
    WHERE match_id = '{MATCH.match_id}' AND action IN ('pass', 'cross') AND outcome_success
      AND sqrt(pow(end_y * {PITCH_W} / 100 - {PITCH_W / 2}, 2)
             + pow({PITCH_L} - end_x * {PITCH_L} / 100, 2)) <= {R}
      AND sqrt(pow(y * {PITCH_W} / 100 - {PITCH_W / 2}, 2)
             + pow({PITCH_L} - x * {PITCH_L} / 100, 2)) > {R}
""")

fig, axes = plt.subplots(1, 2, figsize=(13, 8.5))
for ax, (tid, name, c) in zip(axes, sides):
    draw_pitch(ax)
    ax.set_ylim(26, PITCH_L + 3)   # показываем ~2/3 поля
    ax.add_patch(mpatches.Arc((PITCH_W/2, PITCH_L), 2*R, 2*R, theta1=180, theta2=360,
                              ls="--", color="#7c8aa5", lw=1.3))
    d = de[de.team_id == tid]
    for _, r in d.iterrows():
        ax.annotate("", xy=(r.end_y/100*PITCH_W, r.end_x/100*PITCH_L),
                    xytext=(r.y/100*PITCH_W, r.x/100*PITCH_L), zorder=4,
                    arrowprops=dict(arrowstyle="->", color="#c7d0de", lw=1.2, alpha=0.7,
                                    ls="--" if r.action == "cross" else "-",
                                    shrinkA=0, shrinkB=0))
    s = sh2[sh2.team_id == tid]
    goals, miss = s[s.is_goal], s[~s.is_goal]
    ax.scatter(miss.wy*PITCH_W, miss.lx*PITCH_L, s=miss.xg*650+25,
               facecolor=c, edgecolor=BG, lw=0.6, alpha=0.8, zorder=5)
    ax.scatter(goals.wy*PITCH_W, goals.lx*PITCH_L, s=goals.xg*900+150, marker="*",
               facecolor=MAGENTA, edgecolor="white", lw=0.8, zorder=6)
    ax.set_title(name, color=c)
    ax.text(PITCH_W/2, 27.5, f"{len(s)} ударов  ·  {len(d)} проникновений",
            ha="center", va="bottom", fontsize=10.5, color=c)

fig.suptitle(f"Удары и проникновения · {TITLE_MATCH}", fontsize=15, weight="bold", y=0.98)
fig.text(0.5, 0.015,
         "★ — гол · кружок — удар (размер = xG) · стрелки — успешные передачи в зону ≤20 м от ворот "
         "(пас — сплошная, кросс — пунктир)",
         ha="center", fontsize=9, color=MUT)
plt.tight_layout(rect=[0, 0.03, 1, 0.96]); plt.show()

## 11. Визуализация 7 — досье игрока в матче
Слева — все передачи игрока в выбранном матче (жёлтые — точные, маджента пунктиром — нет).
Справа — где игрок на фоне лиги: **перцентили** по 9 метрикам сезона среди игроков той же
группы позиций с ≥900 минут. Точка — перцентиль (0 = худший, 100 = лучший), под ней —
сырое значение. Игрок задан в параметрах (`PLAYER = ""` — лучший бомбардир команды).

In [ ]:
# --- передачи игрока в матче ---
pp = q(f"""
    SELECT x, y, end_x, end_y, outcome_success
    FROM iceberg.gold.fct_event
    WHERE match_id = '{MATCH.match_id}' AND player_id = '{PLAYER_ROW.player_id}'
      AND action IN ('pass', 'cross') AND end_x IS NOT NULL
""")

# --- пул сравнения: та же группа позиций, ≥900 минут за сезон ---
GROUPS = {
    "вратарей":       ("GK",),
    "защитников":     ("CB", "DF", "RB", "LB", "RWB", "LWB"),
    "полузащитников": ("CM", "MF", "CDM", "LM", "RM", "DM"),
    "атакующих":      ("ST", "FW", "CF", "SS", "RW", "LW", "CAM"),
}
grp_name, grp_pos = next(((g, p) for g, p in GROUPS.items()
                          if PLAYER_ROW.primary_position in p),
                         ("полузащитников", GROUPS["полузащитников"]))
pool = q(f"""
    SELECT p.player_name, p.primary_position, s.*
    FROM iceberg.gold.fct_player_season_stats s
    JOIN iceberg.gold.dim_player p ON s.player_id = p.player_id
    WHERE s.season = '{SEASON}'
      AND (s.minutes >= 900 OR s.player_id = '{PLAYER_ROW.player_id}')
""")
pool = pool[pool.primary_position.isin(grp_pos) |
            (pool.player_id == PLAYER_ROW.player_id)].copy()

DOS_METRICS = [
    ("Пасы / 90",               "pass_total",               "p90"),
    ("Точность пасов, %",       "pass_pct",                 "rate"),
    ("Ключевые пасы / 90",      "key_passes",               "p90"),
    ("xA / 90",                 "expected_assists",         "p90"),
    ("Удары / 90",              "shots",                    "p90"),
    ("npxG / 90",               "non_penalty_xg_understat", "p90"),
    ("Касания в штрафной / 90", "touches_in_box",           "p90"),
    ("Дриблинг / 90",           "successful_dribbles",      "p90"),
    ("Отборы / 90",             "tackles_won",              "p90"),
]
for _, col, kind in DOS_METRICS:
    v = pool[col].astype(float)
    pool[col + "_v"] = v if kind == "rate" else v / pool.minutes * 90
    pool[col + "_pct"] = pool[col + "_v"].rank(pct=True) * 100
subj = pool[pool.player_id == PLAYER_ROW.player_id].iloc[0]

def fmt(v):
    return f"{v:.0f}" if v >= 20 else (f"{v:.1f}" if v >= 2 else f"{v:.2f}")

# --- рисуем ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 8.5),
                               gridspec_kw={"width_ratios": [1, 1.2]})
draw_pitch(ax1)
ok = int(pp.outcome_success.sum())
for _, r in pp.iterrows():
    good = bool(r.outcome_success)
    ax1.annotate("", xy=(r.end_y/100*PITCH_W, r.end_x/100*PITCH_L),
                 xytext=(r.y/100*PITCH_W, r.x/100*PITCH_L), zorder=5 if good else 4,
                 arrowprops=dict(arrowstyle="->", color=YELLOW if good else MAGENTA,
                                 lw=1.5 if good else 1.1, alpha=0.9 if good else 0.6,
                                 ls="-" if good else "--", shrinkA=0, shrinkB=0))
ax1.set_title(f"Передачи в матче: {ok} / {len(pp)} точных" if len(pp)
              else "в этом матче передач нет")

N = len(DOS_METRICS)
ax2.axis("off")
ax2.set_xlim(-4, 185); ax2.set_ylim(-0.7, N)
for xx in (0, 50, 100):
    ax2.text(xx, N - 0.45, str(xx), ha="center", fontsize=7.5, color=MUT)
for i, (lab, col, kind) in enumerate(DOS_METRICS):
    yv = N - 1 - i
    pct, val = float(subj[col + "_pct"]), float(subj[col + "_v"])
    ax2.plot([0, 100], [yv, yv], color="#2a3852", lw=3.5, solid_capstyle="round", zorder=2)
    ax2.scatter([pct], [yv], s=280, facecolor=YELLOW, edgecolor=BG, lw=1.5, zorder=5)
    ax2.text(pct, yv - 0.33, fmt(val), ha="center", va="top", fontsize=9, color=FG,
             bbox=dict(boxstyle="round,pad=0.25", facecolor="#1b2739", edgecolor="none"))
    ax2.text(106, yv, lab, ha="left", va="center", fontsize=11, color=FG)
ax2.set_title(f"На фоне {grp_name} лиги, {SEASON_LBL}")

fig.suptitle(f"{PLAYER_ROW.player_name} · {TITLE_MATCH}", fontsize=15, weight="bold", y=0.99)
fig.text(0.5, 0.015,
         f"справа: точка — перцентиль среди {grp_name} с ≥900 мин, под точкой — значение метрики",
         ha="center", fontsize=9, color=MUT)
plt.tight_layout(rect=[0, 0.03, 1, 0.97]); plt.show()

## 12. Визуализация 8 — pizza-график игрока (перцентили)
Визитка StatsBomb: каждый «кусок» — метрика, длина = **перцентиль** игрока среди сверстников
(атакующие с ≥900 мин). 100 = лучший в лиге, 50 = средний. Цвет: красный — атака,
синий — созидание, зелёный — владение. Данные — `fct_player_season_stats` (~90 метрик).
Игрок — из параметров (если он не атакующий, возьмём лучшего бомбардира лиги).

In [ ]:
ATT = ("ST", "FW", "CF", "SS", "RW", "LW", "CAM")
raw = q(f"""
    SELECT p.player_name, p.primary_position, s.*
    FROM iceberg.gold.fct_player_season_stats s
    JOIN iceberg.gold.dim_player p ON s.player_id = p.player_id
    WHERE s.season = '{SEASON}'
""")
peer = raw[(raw.minutes >= 900) & (raw.primary_position.isin(ATT))].copy()
names = dict(q("SELECT team_id, short_name FROM iceberg.gold.dim_team").values)

# (подпись, колонка, тип, группа): p90 — на 90 минут, rate — уже проценты
METRICS = [
    ("Голы\nбез пен./90",  "non_penalty_goals",        "p90",  "att"),
    ("npxG/90",             "non_penalty_xg_understat", "p90",  "att"),
    ("Удары/90",            "shots",                    "p90",  "att"),
    ("Реализация %",        "goal_conversion_pct",      "rate", "att"),
    ("xA/90",               "expected_assists",         "p90",  "cre"),
    ("Ключевые\nпасы/90",  "key_passes",               "p90",  "cre"),
    ("Моменты/90",          "chances_created",          "p90",  "cre"),
    ("Касания\nв штр./90", "touches_in_box",           "p90",  "cre"),
    ("Дриблинг/90",         "successful_dribbles",      "p90",  "pos"),
    ("Точность\nпасов %",  "pass_pct",                 "rate", "pos"),
]
GROUP_COLOR = {"att": RED, "cre": BLUE, "pos": GREEN}

for _, col, kind, _ in METRICS:
    v = peer[col].astype(float)
    peer[col + "_v"] = v if kind == "rate" else v / peer.minutes * 90
    peer[col + "_pct"] = peer[col + "_v"].rank(pct=True) * 100

# субъект: игрок тура, если он в пуле атакующих; иначе — лучший бомбардир лиги
cand = peer[peer.player_id == PLAYER_ROW.player_id]
if len(cand):
    subj = cand.iloc[0]
else:
    subj = peer.sort_values("goals", ascending=False).iloc[0]
    print(f"{PLAYER_ROW.player_name} не в пуле атакующих (≥900 мин) — показываю {subj.player_name}")
pcts = [float(subj[c + "_pct"]) for _, c, _, _ in METRICS]
vals = [float(subj[c + "_v"]) for _, c, _, _ in METRICS]
labs = [m[0] for m in METRICS]
cols = [GROUP_COLOR[m[3]] for m in METRICS]

N = len(METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
width = 2 * np.pi / N * 0.92
fig = plt.figure(figsize=(9.5, 10))
ax = fig.add_subplot(111, polar=True)
ax.set_facecolor(BG)
ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
ax.bar(angles, pcts, width=width, color=cols, edgecolor=BG, lw=1.5, alpha=0.92, zorder=2)
ax.set_ylim(0, 100)
ax.set_yticks([25, 50, 75]); ax.set_yticklabels(["25", "50", "75"], color=MUT, fontsize=8)
ax.set_xticks(angles); ax.set_xticklabels([])
ax.spines["polar"].set_visible(False)
ax.grid(color="#2a3852")
for ang, lab, pct in zip(angles, labs, pcts):
    ax.text(ang, 112, lab, ha="center", va="center", fontsize=8.5, color=FG)
    ax.text(ang, pct - 9 if pct > 20 else pct + 7, f"{pct:.0f}", ha="center", va="center",
            fontsize=9, color="white" if pct > 20 else MUT, weight="bold")
fig.suptitle(f"{subj.player_name} · {names.get(subj.team_id, subj.team_id)} · {SEASON_LBL}",
             fontsize=15, weight="bold", y=0.97)
fig.text(0.5, 0.915,
         f"перцентиль среди атакующих ≥900 мин ({len(peer)} игроков) · число в куске = перцентиль",
         ha="center", fontsize=10, color=MUT)
handles = [mpatches.Patch(color=GROUP_COLOR[k], label=l) for k, l in
           [("att", "атака"), ("cre", "созидание"), ("pos", "владение")]]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.07),
          ncol=3, fontsize=11)
fig.subplots_adjust(top=0.87, bottom=0.10)
plt.show()

# сводка: сырое значение метрики и перцентиль игрока
pd.DataFrame({
    "метрика": [m[0].replace("\n", " ") for m in METRICS],
    "значение": [round(v, 2) for v in vals],
    "перцентиль": [round(p) for p in pcts],
})

## 13. Визуализация 9 — форма команд: старт сезона → последние туры
«Комета» каждой команды показывает дрейф формы за сезон: **хвост** — средние показатели
в стартовых турах, **голова** — в последних 10. Вправо-вверх — команда и создаёт больше
соперников (xG-разница), и забивает больше (разница голов). Данные: `fct_shot` (xG по
матчам) + `dim_match` (туры и счёт).

In [ ]:
gwd = q(f"""
    WITH tm AS (
        SELECT match_id, gameweek, home_team_id AS team_id, away_team_id AS opp_id,
               home_score AS gf, away_score AS ga
        FROM iceberg.gold.dim_match WHERE season = '{SEASON}' AND is_completed
        UNION ALL
        SELECT match_id, gameweek, away_team_id, home_team_id, away_score, home_score
        FROM iceberg.gold.dim_match WHERE season = '{SEASON}' AND is_completed
    ),
    x AS (SELECT match_id, team_id, sum(xg) AS xg FROM iceberg.gold.fct_shot
          WHERE season = '{SEASON}' GROUP BY 1, 2)
    SELECT t.team_id, t.gameweek, t.gf, t.ga,
           coalesce(xf.xg, 0) AS xg_for, coalesce(xa.xg, 0) AS xg_against
    FROM tm t
    LEFT JOIN x xf ON xf.match_id = t.match_id AND xf.team_id = t.team_id
    LEFT JOIN x xa ON xa.match_id = t.match_id AND xa.team_id = t.opp_id
""")
LAST_N = 10
cutoff = int(gwd.gameweek.max()) - LAST_N
gwd["gd"], gwd["xgd"] = gwd.gf - gwd.ga, gwd.xg_for - gwd.xg_against
names = dict(q("SELECT team_id, short_name FROM iceberg.gold.dim_team").values)

def abbr(name):
    ws = str(name).split()
    return (ws[0][0] + ws[1][:2]).upper() if len(ws) > 1 else str(name)[:3].upper()

pts = []
for tid, d in gwd.groupby("team_id"):
    old, new = d[d.gameweek <= cutoff], d[d.gameweek > cutoff]
    if len(old) >= 3 and len(new) >= 3:
        pts.append((tid, old.xgd.mean(), old.gd.mean(), new.xgd.mean(), new.gd.mean()))

COMET_COLORS = [BLUE, RED, TEAL, YELLOW, MAGENTA, GREEN, "#9b6bff", "#ff9f45",
                "#5ad1e6", "#e66ad2", "#a3e635", "#f97316", "#60a5fa", "#f43f5e",
                "#34d399", "#facc15", "#c084fc", "#fb7185", "#4ade80", "#38bdf8"]

fig, ax = plt.subplots(figsize=(11, 9))
for (tid, x0, y0, x1, y1), c in zip(pts, COMET_COLORS * 2):
    ts = np.linspace(0, 1, 40)
    xs, ys = x0 + (x1 - x0) * ts, y0 + (y1 - y0) * ts
    for i in range(len(ts) - 1):
        ax.plot(xs[i:i+2], ys[i:i+2], color=c, alpha=0.06 + 0.5 * ts[i],
                lw=0.8 + 3.6 * ts[i], solid_capstyle="round", zorder=2)
    ax.scatter([x1], [y1], s=620, facecolor=BG, edgecolor=c, lw=2, zorder=4)
    ax.text(x1, y1, abbr(names.get(tid, "?")), ha="center", va="center",
            fontsize=7.5, color=FG, weight="bold", zorder=5)
ax.axhline(0, ls="--", color="#3a4761", lw=1); ax.axvline(0, ls="--", color="#3a4761", lw=1)
ax.set_title(f"АПЛ {SEASON_LBL}: форма — туры 1–{cutoff} → последние {LAST_N} туров")
ax.set_xlabel("xG-разница за матч"); ax.set_ylabel("разница голов за матч")
ax.margins(0.08)
fig.text(0.5, 0.015, "голова кометы — последние туры, хвост — старт сезона",
         ha="center", fontsize=9, color=MUT)
plt.tight_layout(rect=[0, 0.03, 1, 1]); plt.show()

## 14. Визуализация 10 — география паса: команда vs лига
Поле делится на 9 зон; в каждой — доля пасов команды, **начатых** из этой зоны,
и отклонение от среднего по лиге (в процентных пунктах). Зелёное — команда играет через
эту зону чаще лиги, красное — реже. Агрегация сотен тысяч пасов сезона — целиком в Trino,
в ноутбук приезжает 18 чисел.

In [ ]:
zc = q(f"""
    SELECT if(team_id = '{TEAM_ID}', 1, 0) AS is_team,
           least(cast(floor(x / 33.4) AS int), 2) AS zl,
           least(cast(floor(y / 33.4) AS int), 2) AS zw,
           count(*) AS passes
    FROM iceberg.gold.fct_event
    WHERE season = '{SEASON}' AND action = 'pass'
    GROUP BY 1, 2, 3
""")
tz = zc[zc.is_team == 1].set_index(["zl", "zw"]).passes
lz = zc.groupby(["zl", "zw"]).passes.sum()
tsh, lsh = tz / tz.sum() * 100, lz / lz.sum() * 100

fig, ax = plt.subplots(figsize=(7.2, 10))
draw_pitch(ax)
for zl in range(3):
    for zw in range(3):
        t = float(tsh.get((zl, zw), 0)); d = t - float(lsh.get((zl, zw), 0))
        color = GREEN if d >= 0 else RED
        ax.add_patch(plt.Rectangle((zw * PITCH_W/3, zl * PITCH_L/3), PITCH_W/3, PITCH_L/3,
                     facecolor=color, alpha=min(abs(d)/5, 1)*0.5 + 0.05,
                     edgecolor=BG, lw=2, zorder=1))
        cx, cy = zw*PITCH_W/3 + PITCH_W/6, zl*PITCH_L/3 + PITCH_L/6
        ax.text(cx, cy + 2.4, f"{t:.0f}%", ha="center", va="center",
                fontsize=15, weight="bold", color=FG, zorder=5)
        ax.text(cx, cy - 2.8, f"{d:+.1f} пп", ha="center", va="center", fontsize=10,
                color="#a7f3d0" if d >= 0 else "#fecdd3", zorder=5)
ax.set_title(f"{TEAM} {SEASON_LBL}: откуда начинаются пасы — vs средняя по лиге")
fig.text(0.5, 0.02, "в зоне: доля пасов команды / отклонение от лиги, пп · атака вверх",
         ha="center", fontsize=9, color=MUT)
plt.tight_layout(rect=[0, 0.03, 1, 1]); plt.show()

## 15. Визуализация 11 — классы передач (k-means на numpy)
Все пасы команды за сезон кластеризуются по началу и концу (4 признака) — обычный k-means,
написанный в 10 строк на numpy, без sklearn. Жирная стрелка — «средний» пас кластера,
бейдж — его точность, тонкие стрелки — случайные примеры. Видны фирменные маршруты
команды: развитие через фланги, диагонали, забросы за спину.

In [ ]:
ps = q(f"""
    SELECT x, y, end_x, end_y, outcome_success
    FROM iceberg.gold.fct_event
    WHERE season = '{SEASON}' AND team_id = '{TEAM_ID}' AND action = 'pass'
""")
sx, sy = ps.y.values/100*PITCH_W, ps.x.values/100*PITCH_L   # старт паса
ex, ey = ps.end_y.values/100*PITCH_W, ps.end_x.values/100*PITCH_L
DW = 1.7  # вес направления: кластеры различают не только «откуда», но и «куда»
X = np.column_stack([sx, sy, (ex - sx) * DW, (ey - sy) * DW])

def kmeans(X, k, iters=40, seed=7):
    """Наивный k-means: достаточно для демо, sklearn не нужен."""
    rng = np.random.default_rng(seed)
    C = X[rng.choice(len(X), k, replace=False)]
    for _ in range(iters):
        lab = ((X[:, None, :] - C[None]) ** 2).sum(-1).argmin(1)
        C = np.stack([X[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(k)])
    return lab, C

K, TOP = 14, 9
lab, C = kmeans(X, K)
order = np.argsort(-np.bincount(lab, minlength=K))[:TOP]  # крупнейшие кластеры

CLUSTER_COLORS = [BLUE, RED, TEAL, YELLOW, MAGENTA, GREEN, "#9b6bff", "#ff9f45", "#5ad1e6"]
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(8.5, 12))
draw_pitch(ax)
for j, c in zip(order, CLUSTER_COLORS):
    idx = np.where(lab == j)[0]
    for i in rng.choice(idx, size=min(30, len(idx)), replace=False):
        ax.annotate("", xy=(ex[i], ey[i]), xytext=(sx[i], sy[i]), zorder=3,
                    arrowprops=dict(arrowstyle="->", color=c, lw=0.7, alpha=0.2,
                                    shrinkA=0, shrinkB=0))
    cx, cy = C[j, 0], C[j, 1]                       # средний старт кластера
    cdx, cdy = C[j, 2] / DW, C[j, 3] / DW           # средний вектор паса
    ax.annotate("", xy=(cx + cdx, cy + cdy), xytext=(cx, cy), zorder=6,
                arrowprops=dict(arrowstyle="-|>", color=c, lw=3.2, shrinkA=0, shrinkB=0))
    acc = 100 * ps.outcome_success.values[idx].mean()
    ax.text(cx, cy, f"{acc:.0f}%", fontsize=8.5, weight="bold", color=BG,
            ha="center", va="center", zorder=7,
            bbox=dict(boxstyle="round,pad=0.25", facecolor=c, edgecolor="none"))

n_lbl = f"{len(ps):,}".replace(",", " ")
ax.set_title(f"{TEAM} {SEASON_LBL}: классы передач · k-means по {n_lbl} пасам")
fig.text(0.5, 0.02,
         f"жирная стрелка — средний пас кластера · бейдж — точность пасов кластера · "
         f"тонкие — 30 случайных примеров · показаны {TOP} из {K} кластеров",
         ha="center", fontsize=9, color=MUT)
plt.tight_layout(rect=[0, 0.03, 1, 1]); plt.show()

## Дальше
- Поменяй параметры в начале — сезон, команду, соперника, игрока — и прогони ноутбук
  заново: все 11 графиков пересчитаются.
- Описания колонок — в каталоге [meta.sk-vpn-2026.uk](https://meta.sk-vpn-2026.uk).
- Ещё идеи из этих же данных: **сеть пасов матча** (`fct_event`: кто кому),
  **bump-график гонки за титул** (`fct_standings` / `fct_team_elo`),
  **сравнение судей** (`dim_referee` + фолы из `fct_event`).
- Хочешь, чтобы запросы шли под твоим именем (видно в истории Trino) — см.
  `docs/ANALYST_ONBOARDING.md`, вариант с `OAuth2Authentication`.
- Данные только для чтения: любой INSERT/UPDATE Trino отклонит.